[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/tabular-ml-practice/02_preprocessing/02_preprocessing.ipynb)

# 02. 전처리 — 지저분한 데이터를 모델이 먹을 수 있게 만들기

[01_eda_visualization](../01_eda_visualization/01_eda_visualization.ipynb)에서 데이터를 살펴보고
"무엇을 고쳐야 하는가"의 목록을 만들었습니다. 이 노트북에서 그 목록을 하나씩 처리합니다.

전처리의 목적은 세 가지입니다.

1. **모델이 읽을 수 있는 형태로 만들기** — scikit-learn과 [Keras](../../../glossary.md#keras)는 **숫자 배열만** 받습니다.
   문자열이 있거나 빈 칸이 있으면 그 자리에서 에러가 납니다.
2. **학습을 방해하는 값 걷어내기** — [이상치](../../../glossary.md#outlier) 몇 개가 전체 학습을 끌고 갈 수 있습니다.
3. **공정하게 평가할 수 있게 나누기** — 학습에 쓴 데이터로 점수를 매기면 아무 의미가 없습니다.

## 이 노트북의 구성

**1부에서는 택시 데이터로 전처리를 끝까지 한 바퀴 돌립니다.** 이상치를 걷어내고, [결측치](../../../glossary.md#missing-value)를
처리하고, 범주형을 인코딩하고, 학습·검증으로 나눠 [스케일링](../../../glossary.md#scaling)하는 순서입니다. 여기서 만나는 것은
답이 분명한 경우들입니다 — 시속 4,834마일은 지우면 되고, 결측 1%도 지우면 됩니다.

**2부의 타이타닉에서는 판단이 필요해집니다.** 512파운드짜리 특실 요금은 지워야 할까요?
`age`의 20%가 비어 있다면 채워야 할까요, 그 행을 버려야 할까요?

| | 데이터 | 다루는 것 |
|---|---|---|
| **1부** | `trips` (회귀) | 이상치(도메인 규칙) → 결측치(삭제) → 컬럼 정리 → 인코딩 → 분리 → 스케일링 |
| **2부** | `titanic` (분류) | 순서를 틀렸을 때, [IQR](../../../glossary.md#iqr) 기준, 결측치 **대체**, `stratify` |

| 1부에서는 | 2부에서는 |
|---|---|
| "시속 4,834마일"처럼 **답이 정해진** 이상치 | 512파운드 특실 요금 — **지울지 말지 판단이 필요** |
| 결측 1%대 → 그냥 삭제 | 결측 20~77% → **채울까 버릴까** |
| 회귀라 `stratify`가 필요 없음 | 분류라 **클래스 비율 유지**가 필요 |

1부 끝에서 `prepare_trips()`, 2부 끝에서 `prepare_titanic()` 함수를 완성합니다.
**03·04번 노트북에서 이 두 함수를 그대로 가져다 씁니다.**

## 이 노트북을 읽는 법

- **셀을 위에서부터 순서대로 실행하세요** (`Shift + Enter`). 아래쪽 셀은 위쪽 셀에서 만든
  변수를 그대로 쓰기 때문에, 중간부터 실행하면 `NameError`가 납니다.
- **실행 결과는 저장되어 있지 않습니다.** 코드 셀 아래가 비어 있는 것이 정상이고,
  직접 실행해야 표와 그래프가 나타납니다.
- 본문에 적힌 숫자(예: "MAE 8.33분")는 **실행하면 나오는 값**입니다. 글을 읽으면서
  그 숫자가 어느 셀의 출력인지 짚어보면 이해가 빠릅니다.
- 코드는 **그대로 실행만 해도 되지만**, 숫자를 바꿔 다시 실행해보는 것이 가장 좋은 연습입니다.
- **pandas 문법이 막히면** [00_pandas_for_tabular](../00_pandas_for_tabular/00_pandas_for_tabular.ipynb)에
  이 시리즈에서 쓰는 문법만 모아뒀습니다(`pd.to_datetime`, `get_dummies`, `dropna` …). 사전처럼 찾아보세요.
- 낯선 용어는 [glossary.md](../../../glossary.md)에서 찾아보세요.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

if IN_COLAB:
    !pip install -q pandas seaborn matplotlib scikit-learn koreanize-matplotlib

### 준비 셀 — 라이브러리와 한글 폰트

아래 셀은 네 노트북에 공통으로 들어가는 준비 코드입니다. **내용을 이해할 필요는 없고 그냥
실행**하면 됩니다. `numpy`·`pandas`·`matplotlib`·`seaborn`을 불러오고, 그래프의 한글이
깨지지 않게 폰트를 잡고, 결과가 매번 같도록 무작위 시드(`RANDOM_STATE = 42`)를 고정합니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# 그래프에 한글이 깨지지 않도록 폰트를 설정합니다.
# koreanize-matplotlib이 있으면 그걸 쓰고, 없으면 OS에 설치된 한글 폰트를 찾습니다.
try:
    import koreanize_matplotlib  # noqa: F401
except ImportError:
    import matplotlib.font_manager as fm

    for _name in ["Malgun Gothic", "AppleGothic", "NanumGothic"]:
        if any(_name == f.name for f in fm.fontManager.ttflist):
            plt.rc("font", family=_name)
            break
plt.rcParams["axes.unicode_minus"] = False  # 한글 폰트에서 마이너스 기호가 깨지는 것 방지

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42

### 데이터 불러오기

01번 1부와 같은 코드입니다. 원본 컬럼에서 예측 대상인 `duration`(이동 시간)과
[파생 변수](../../../glossary.md#feature-engineering) `speed`·`weekday`·`hour`를 만들어 둡니다.

In [ ]:
trips = sns.load_dataset("taxis")
trips["pickup"] = pd.to_datetime(trips["pickup"])
trips["dropoff"] = pd.to_datetime(trips["dropoff"])
trips["duration"] = (trips["dropoff"] - trips["pickup"]).dt.total_seconds() / 60
trips["speed"] = trips["distance"] / (trips["duration"] / 60)
trips["weekday"] = trips["pickup"].dt.dayofweek
trips["hour"] = trips["pickup"].dt.hour

print("trips:", trips.shape)

---

# 1부. 택시 데이터로 전처리 한 바퀴

`trips` 하나만 봅니다. 01번 1부에서 만든 계획표가 그대로 작업 목록입니다.

| 관찰 | 처리 |
|---|---|
| `speed` max = 4834, `duration` min = 0 | 이상치 행 제거 |
| 결측치 1%대 | 해당 행 제거 |
| `speed`(누출) · `total`(중복) · `pickup_zone`(194개) · datetime | 컬럼 삭제 |
| `color`, `payment`, `*_borough`가 문자열 | [원-핫 인코딩](../../../glossary.md#one-hot-encoding) |
| 모델 학습 준비 | 학습/검증 분리 → 스케일링 |

타이타닉은 2부에서 불러옵니다.

---

## 1. 순서가 중요하다

전처리는 단계마다 독립적이지 않습니다. **순서를 바꾸면 결과가 달라집니다.**

```
  원본 데이터
      │
  ① 이상치 제거          ← 잘못된 행을 먼저 걷어낸다
      │
  ② 불필요한 컬럼 삭제    ← 결측치 처리보다 반드시 먼저!
      │
  ③ 결측치 처리
      │
  ④ 범주형 인코딩         ← 분리보다 먼저 (컬럼 구성을 맞추기 위해)
      │
  ⑤ 학습/검증 분리        ← 여기서부터 검증 데이터는 "없는 것"처럼 취급
      │
  ⑥ 스케일링             ← fit은 학습 데이터에만!
      │
  모델 학습
```

순서에 대한 이유가 세 군데 있습니다.

- **② 컬럼 삭제가 ③ 결측치보다 먼저**: 결측치가 잔뜩 있는 컬럼을 남겨둔 채 `dropna()`를 하면,
  그 컬럼 때문에 멀쩡한 행까지 대량으로 날아갑니다. (2부에서 타이타닉으로 실제 확인합니다)
- **④ 인코딩이 ⑤ 분리보다 먼저**: 나눈 뒤에 각각 인코딩하면 한쪽에만 있는 범주 때문에
  컬럼 구성이 달라집니다.
- **⑥ 스케일링이 ⑤ 분리보다 나중**: 나누기 전에 스케일링하면 검증 데이터의 정보가
  학습 과정에 새어 들어갑니다. (7절에서 자세히)

---

## 2. 이상치 (Outlier)

### 이상치에는 두 종류가 있습니다

| 종류 | 예 | 처리 |
|---|---|---|
| **기록 오류** | 9.4마일을 7초에 이동, 나이 999세 | 물리적으로 불가능 → **제거** |
| **실제 극단값** | 타이타닉 1등급 특실 요금 512파운드 | 진짜 있었던 값 → **판단 필요** *(2부)* |

**둘을 구분하지 않고 "통계적으로 튀니까 지운다"고 하면 위험합니다.** 사기 탐지에서는 이상치가
바로 찾으려는 대상이고, 고가 상품 매출 예측에서 극단값을 지우면 정작 중요한 구간을 못 배웁니다.

이상치를 걸러내는 방법은 두 가지입니다. **① 도메인 규칙**은 여기서, **② IQR 기준**은
답이 딱 떨어지지 않는 경우라 2부에서 다룹니다.

### 방법 ① 도메인 규칙 — "이건 말이 안 된다"

가장 확실한 방법입니다. 해당 분야의 상식으로 불가능한 값을 걸러냅니다.
`trips`에서는 두 가지가 명백합니다.

- `duration <= 0`: 이동 시간이 0 이하 — 6건
- `speed >= 60`: 뉴욕 시내 택시가 시속 60마일 이상 — 11건

In [ ]:
print("duration <= 0 :", (trips["duration"] <= 0).sum(), "건")
print("speed >= 60   :", (trips["speed"] >= 60).sum(), "건")
print("둘 중 하나     :", ((trips["duration"] <= 0) | (trips["speed"] >= 60)).sum(), "건")

이제 두 조건을 **뒤집어 정상 행만 남깁니다.** `df[조건]`은 조건이 `True`인 행만 골라내고,
`&`는 "두 조건을 모두 만족"을 뜻합니다.

In [ ]:
# 조건에 맞는 행만 남기는 방식(불리언 인덱싱)
trips_clean = trips[(trips["duration"] > 0) & (trips["speed"] < 60)].copy()

print(f"{len(trips)} -> {len(trips_clean)} 행 ({len(trips) - len(trips_clean)}건 제거)")

> #### `.copy()`를 붙이는 이유
>
> `trips[조건]`은 원본의 일부를 잘라낸 **뷰(view)** 일 수 있습니다. 여기에 값을 대입하면
> pandas가 `SettingWithCopyWarning`을 띄우고, 원본이 바뀌는지 사본이 바뀌는지 보장되지 않습니다.
>
> ```python
> df_temp = trips[trips["speed"] < 60]      # 뷰일 수 있음
> df_temp.drop("fare", axis=1, inplace=True)  # ⚠️ SettingWithCopyWarning
> ```
>
> **필터링한 결과를 계속 수정할 거라면 `.copy()`를 붙이는 습관**을 들이면 이 문제를 아예 피할 수 있습니다.
>
> 조건을 여러 개 걸 때는 각 조건을 **괄호로 감싸야** 합니다. `&`(and)와 `|`(or)가 비교 연산자보다
> 우선순위가 높기 때문에, `trips["duration"] > 0 & trips["speed"] < 60`은 엉뚱하게 해석됩니다.

---

## 3. 결측치 (Missing Value)

### 왜 반드시 처리해야 하나

scikit-learn 모델 대부분은 `NaN`이 들어오면 **에러를 냅니다.** 그래서 반드시 처리해야 합니다.
(실제 에러 화면은 2부에서 결측이 많은 데이터로 확인합니다.)

처리 방법은 **삭제**와 **대체** 두 가지입니다. 택시는 결측 비율이 낮아 삭제로 끝나므로
여기서는 삭제만 보고, **대체는 2부**에서 다룹니다.

### 방법 ① 삭제 — `dropna()`

가장 간단합니다. 결측치가 **전체의 5% 미만**이고 특정 집단에 몰려 있지 않다면 무난한 선택입니다.

```python
df.dropna()                        # 한 칸이라도 비면 행 삭제
df.dropna(subset=["age"])          # age가 빈 행만 삭제
df.dropna(axis=1)                  # 결측치가 있는 "컬럼"을 삭제
df.dropna(thresh=10)               # 값이 10개 미만인 행만 삭제
```

In [ ]:
print("trips_clean 결측치")
print(trips_clean.isnull().sum()[lambda s: s > 0])
print()
print(f"전체 {len(trips_clean)}행 중 결측치가 있는 행: "
      f"{trips_clean.isnull().any(axis=1).sum()}행 "
      f"({trips_clean.isnull().any(axis=1).mean() * 100:.1f}%)")

`trips`의 결측치는 지역·결제 수단 컬럼에 있고, 영향받는 행은 전체의 **1.2%** 뿐입니다.
이 정도면 삭제해도 무방합니다.

> `dropna`의 옵션(`subset`, `axis`)과 `fillna`의 사용법은 [00번 4절](../00_pandas_for_tabular/00_pandas_for_tabular.ipynb)에 정리돼 있습니다.

---

## 4. 컬럼 정리

01번에서 정리한 계획표대로, 다음 네 종류의 컬럼을 걷어냅니다.

| 종류 | `trips`에서 | 왜 |
|---|---|---|
| **[데이터 누출](../../../glossary.md#data-leakage)** | `speed` | 정답(`duration`)으로 계산된 값 — 넣으면 모델이 정답을 그대로 복원 |
| **중복 정보** | `total` (= `fare`+`tip`+`tolls`) | 같은 정보를 두 번 넣을 필요 없음 |
| **고유값 폭발** | `pickup_zone` (194개) | 인코딩하면 컬럼이 수백 개로 늘어남 |
| **그대로 쓸 수 없는 형** | `pickup`, `dropoff` (datetime) | `weekday`/`hour`로 이미 뽑아냄 |

In [ ]:
trips_clean = trips_clean.drop(columns=[
    "pickup", "dropoff",              # datetime — weekday/hour로 대체함
    "pickup_zone", "dropoff_zone",    # 고유값 194/203개
    "speed",                          # duration으로 계산한 값 → 데이터 누출
    "total",                          # fare + tip + tolls → 중복
])

print(trips_clean.shape)
print(list(trips_clean.columns))

컬럼을 먼저 정리했으니 이제 결측치를 처리합니다. 택시는 지운 컬럼에 결측이 몰려 있지 않아
순서를 바꿔도 결과가 비슷하지만, **이 순서를 지키는 습관이 2부에서 431행의 차이**를 만듭니다.

In [ ]:
print("삭제 전:", trips_clean.shape)
trips_clean = trips_clean.dropna()
print("삭제 후:", trips_clean.shape)

---

## 5. 범주형 인코딩

### 왜 필요한가

모델은 숫자만 계산할 수 있습니다. `"Manhattan"`이라는 문자열은 더하거나 곱할 수 없습니다.
그래서 **범주를 숫자로 바꿔야** 하는데, 방식이 두 가지 있고 **아무거나 쓰면 안 됩니다.**

### 방식 ① 레이블 인코딩 — 순서가 있을 때만

각 범주에 0, 1, 2… 번호를 매깁니다.

```
Manhattan → 0,  Queens → 1,  Brooklyn → 2,  Bronx → 3
```

간단하지만 **모델은 이 숫자를 크기로 해석합니다.** "Bronx(3) > Brooklyn(2)"이고
"Manhattan + Queens = Brooklyn"이라는, 원래 없던 관계가 만들어집니다.

**순서가 실제로 있는 범주에만** 써야 합니다.

- ✅ 학점 (`F` < `D` < `C` < `B` < `A`), 등급 (`저` < `중` < `고`)
- ❌ 지역, 색상, 결제 수단 — 순서가 없음

### 방식 ② 원-핫 인코딩 — 순서가 없을 때 (대부분의 경우)

**범주마다 컬럼을 하나씩 만들고 해당하면 1, 아니면 0**을 넣습니다.

| 원본 | → | `Manhattan` | `Queens` | `Brooklyn` |
|---|---|---|---|---|
| Manhattan | | **1** | 0 | 0 |
| Queens | | 0 | **1** | 0 |
| Brooklyn | | 0 | 0 | **1** |

범주 사이에 크기 관계가 생기지 않습니다. pandas에서는 `get_dummies` 한 줄입니다.
(`drop_first`, `select_dtypes`까지 포함한 문법 설명은 [00번 7절](../00_pandas_for_tabular/00_pandas_for_tabular.ipynb))

In [ ]:
# 어떤 컬럼이 인코딩 대상인지 확인
object_cols = trips_clean.select_dtypes(include="object").columns.tolist()
print("object 컬럼:", object_cols)
print()
for c in object_cols:
    print(f"  {c:18s} 고유값 {trips_clean[c].nunique()}개: {list(trips_clean[c].unique())}")

실제로 변환해서 **컬럼이 몇 개로 늘어나는지** 확인해봅시다.

In [ ]:
encoded = pd.get_dummies(trips_clean, columns=object_cols)   # get_dummies: 문자열 범주를 범주마다 0/1 컬럼으로 펼친다(원-핫 인코딩)

print(f"{trips_clean.shape[1]}개 컬럼 -> {encoded.shape[1]}개 컬럼")
print()
print(list(encoded.columns))

`color`(2개) + `payment`(2개) + `pickup_borough`(4개) + `dropoff_borough`(5개) = 13개 컬럼이
새로 생기고 원래 4개 컬럼이 사라져서, 12개 → 21개가 되었습니다.

`columns=`를 지정하지 않으면 **`get_dummies`가 알아서 모든 `object` 컬럼을 변환**합니다.
다만 어떤 컬럼이 변환되는지 명시적으로 드러나는 편이 나중에 읽기 좋습니다.

### `drop_first` — 더미 변수 함정

원-핫 인코딩에는 미묘한 문제가 있습니다. `color`를 봅시다.

| `color_green` | `color_yellow` |
|---|---|
| 1 | 0 |
| 0 | 1 |

**두 컬럼은 항상 합이 1입니다.** 즉 하나만 알면 나머지는 자동으로 결정됩니다.
`color_green`이 0이면 `color_yellow`는 반드시 1입니다.

이렇게 **한 변수가 다른 변수로 완전히 설명되는 상태**를 다중공선성(multicollinearity)이라 하고,
회귀 계열 모델에서 계수 계산을 불안정하게 만듭니다. 이것을 **더미 변수 함정(dummy variable trap)** 이라고 합니다.

해결은 `drop_first=True`로 **각 범주의 첫 컬럼을 하나씩 빼는 것**입니다. 정보는 하나도 잃지 않습니다 —
`color_yellow=0`이 곧 "green"을 뜻하기 때문입니다.

In [ ]:
without = pd.get_dummies(trips_clean, columns=object_cols)
with_drop = pd.get_dummies(trips_clean, columns=object_cols, drop_first=True)

print(f"drop_first=False : {without.shape[1]}개 컬럼")
print(f"drop_first=True  : {with_drop.shape[1]}개 컬럼")
print()
print("빠진 컬럼:", sorted(set(without.columns) - set(with_drop.columns)))

범주마다 첫 번째 것(`color_green`, `payment_cash`, `pickup_borough_Bronx`,
`dropoff_borough_Bronx`)이 빠져 4개 컬럼이 줄었습니다.

- **회귀·신경망**: `drop_first=True` 권장
- **트리 모델**: 다중공선성의 영향을 받지 않아 어느 쪽이든 무방
- **해석이 중요할 때**: `drop_first=False`가 각 범주의 효과를 그대로 보여줘 읽기 편함

이 시리즈는 `drop_first=True`로 통일합니다.

### 인코딩을 분리보다 먼저 하는 이유

01번 연습문제에서 확인했듯, `Staten Island`는 **하차에만 2건 있고 승차에는 없습니다.**
이렇게 드문 범주가 있을 때 **데이터를 먼저 나눈 뒤 각각 인코딩하면 컬럼 구성이 어긋납니다.**

In [ ]:
# 문제 상황 재현: 먼저 나누고 각각 인코딩
from sklearn.model_selection import train_test_split

part_a, part_b = train_test_split(trips_clean, test_size=0.2, random_state=2)

enc_a = pd.get_dummies(part_a, columns=object_cols)
enc_b = pd.get_dummies(part_b, columns=object_cols)

print("A쪽 컬럼 수:", enc_a.shape[1])
print("B쪽 컬럼 수:", enc_b.shape[1])
print("A에만 있는 컬럼:", sorted(set(enc_a.columns) - set(enc_b.columns)))
print("B에만 있는 컬럼:", sorted(set(enc_b.columns) - set(enc_a.columns)))

**A쪽은 21개, B쪽은 20개입니다.** `Staten Island` 2건이 전부 A쪽에 들어가는 바람에
`dropoff_borough_Staten Island` 컬럼이 A에만 생겼습니다.

이 상태로 `model.fit(A)` 후 `model.predict(B)`를 하면
**"학습할 때는 21개 피처였는데 20개가 들어왔다"** 는 에러가 납니다.

> 이 문제는 **나눌 때마다 재현되지도 않습니다.** `random_state`를 바꾸면 두 쪽 다 21개가 나오는
> 경우도 있습니다. 어쩌다 한 번씩 터지는 버그가 가장 잡기 어렵다는 점에서, 애초에 발생하지 않는
> 순서로 짜는 것이 낫습니다.

해결책은 두 가지입니다.

1. **인코딩을 먼저 하고 나중에 나눈다** ← 이 시리즈의 방식. 가장 간단합니다.
2. 나눈 뒤 인코딩했다면 `reindex`로 컬럼을 맞춘다:
   ```python
   enc_b = enc_b.reindex(columns=enc_a.columns, fill_value=0)
   ```

> 실무 파이프라인에서는 scikit-learn의 `OneHotEncoder(handle_unknown="ignore")`를
> `ColumnTransformer`에 넣어 씁니다. 학습 때 본 범주 목록을 기억해두고, 검증·운영 데이터에서
> 처음 보는 범주가 나오면 전부 0으로 처리해줍니다. 새 데이터가 계속 들어오는 서비스라면
> `get_dummies`보다 이쪽이 안전합니다.

In [ ]:
# 이 시리즈의 방식: 인코딩 먼저
trips_encoded = pd.get_dummies(trips_clean, columns=object_cols, drop_first=True)

print("trips:", trips_encoded.shape)
print("남은 object 컬럼:", trips_encoded.select_dtypes(include="object").columns.tolist())
trips_encoded.head(3)

---

## 6. 학습 데이터와 검증 데이터 나누기

### 왜 나누는가

학습에 쓴 데이터로 점수를 매기면, 모델이 답을 통째로 외웠는지 패턴을 배웠는지 구분할 수 없습니다.
**한 번도 보지 못한 데이터에서의 성능**만이 실제 성능입니다.

```python
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
```

| 인자 | 의미 |
|---|---|
| `test_size=0.2` | 검증용 20% (=80:20 분할). 데이터가 적으면 0.3, 아주 많으면 0.1 |
| `random_state=42` | 무작위 섞기의 시드. **고정해야 다시 실행해도 같은 결과** |
| `stratify=y` | 분할 후에도 `y`의 클래스 비율을 유지 (분류 문제에서만, **2부**) |

**반환 순서는 `X_train, X_valid, y_train, y_valid`** 입니다. `X_train, y_train, X_valid, y_valid`로
잘못 받으면 에러 없이 엉뚱한 데이터로 학습하게 되니 주의하세요.

In [ ]:
# [회귀] trips
X = trips_encoded.drop(columns="duration")
y = trips_encoded["duration"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"학습: {X_train.shape}  검증: {X_valid.shape}")

---

## 7. 스케일링

### 왜 필요한가

컬럼마다 값의 범위가 다릅니다. `trips`에서 `distance`는 0~37, `fare`는 1~150,
`hour`는 0~23입니다. 이 차이가 문제가 되는 모델이 있습니다.

| 모델 | 스케일링 필요? | 이유 |
|---|---|---|
| 선형 회귀, [로지스틱 회귀](../../../glossary.md#logistic-regression) | ✅ | [경사 하강법](../../../glossary.md#gradient-descent)이 범위 큰 변수에 끌려감 |
| **신경망** | ✅ **필수** | 위와 같은 이유 + 활성화 함수의 포화 |
| KNN, SVM | ✅ **필수** | 거리 계산에서 범위 큰 변수가 지배 |
| **[결정 트리](../../../glossary.md#decision-tree), [랜덤 포레스트](../../../glossary.md#random-forest)** | ❌ **불필요** | 각 컬럼을 **따로** 보며 "이 값보다 큰가/작은가"만 판단 |

마지막 항목이 중요합니다. 03번의 트리 모델은 스케일링 없이도 잘 동작하고,
04번의 신경망은 스케일링이 없으면 학습이 거의 안 됩니다.

### 세 가지 스케일러

| 스케일러 | 계산 | 결과 | 이상치에 |
|---|---|---|---|
| `StandardScaler` | (x - 평균) / 표준편차 | 평균 0, 표준편차 1 | 민감 |
| `MinMaxScaler` | (x - 최소) / (최대 - 최소) | **0 ~ 1** | **매우 민감** |
| `RobustScaler` | (x - 중앙값) / IQR | 중앙값 0 | **강함** |

`RobustScaler`가 이상치에 강한 이유는 평균·표준편차 대신 **중앙값과 IQR**을 쓰기 때문입니다.
극단값 하나가 평균은 크게 흔들지만 중앙값은 거의 못 움직입니다.

이상치가 있는 `fare` 컬럼으로 직접 비교해봅시다.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

fare = trips[["fare"]].dropna()

results = {}
for name, scaler in [("Standard", StandardScaler()),
                     ("MinMax", MinMaxScaler()),
                     ("Robust", RobustScaler())]:
    results[name] = scaler.fit_transform(fare).ravel()

summary = pd.DataFrame({
    # np.median: 중앙값. Series가 아니라 numpy 배열이라 pandas의 .median() 대신 이것을 쓴다
    name: {"최솟값": v.min(), "중앙값": np.median(v), "최댓값": v.max(), "표준편차": v.std()}
    for name, v in results.items()
}).round(3)
summary

표의 숫자만으로는 감이 잘 안 옵니다. 세 결과의 **분포를 나란히 그려** 비교합니다.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (name, v) in zip(axes, results.items()):
    sns.histplot(v, bins=50, ax=ax)   # histplot: 값을 구간(bin)으로 나눠 개수를 막대로 그린다
    ax.set_title(f"{name}Scaler")
    ax.set_xlabel("변환된 fare")

plt.tight_layout()
plt.show()

**`MinMaxScaler`의 중앙값이 0.057입니다.** 요금 150달러짜리 극단값 하나가 최댓값을 잡아버려서,
정상 요금 대부분이 **0~0.1 구간에 짜부라졌습니다.** 0~1 사이로 예쁘게 들어가긴 했지만
값들 사이의 구분이 사라진 셈입니다.

`RobustScaler`는 중앙값이 정확히 0이고 정상 구간이 넓게 퍼져 있습니다.
**이상치가 있는 데이터라면 `RobustScaler`가 안전한 선택**입니다.

> 원-핫 인코딩으로 만든 0/1 컬럼도 스케일링 대상에 포함됩니다. 이미 0과 1이라 안 해도 될 것
> 같지만, 다른 컬럼과 범위를 맞춰주는 편이 신경망 학습에 유리합니다. 트리 모델이라면
> 애초에 스케일링 자체가 필요 없습니다.

### 가장 중요한 규칙 — `fit`은 학습 데이터에만

```python
scaler = RobustScaler()
X_train = scaler.fit_transform(X_train)   # 학습 데이터로 기준을 "계산하고" 적용
X_valid = scaler.transform(X_valid)       # 같은 기준을 "적용만"
```

- **`fit`**: 데이터를 보고 기준(평균·표준편차·중앙값·IQR)을 계산
- **`transform`**: 이미 계산된 기준으로 값을 변환
- **`fit_transform`**: 둘을 한 번에

**검증 데이터에 `fit`을 하면 안 됩니다.** 검증 데이터의 통계량이 변환 과정에 반영되는 것은,
"한 번도 보지 못한 데이터"라는 전제를 깨는 일입니다. 이것도 **데이터 누출**입니다.

실제 서비스를 생각하면 명확합니다. 사용자 요청이 **한 건씩** 들어올 때 그 한 건의 평균과
표준편차를 계산할 수는 없습니다. 학습 때 저장해둔 기준을 쓰는 것 외에 방법이 없습니다.

In [ ]:
scaler = RobustScaler()

# 올바른 방법
X_train_s = scaler.fit_transform(X_train)
X_valid_s = scaler.transform(X_valid)

# 잘못된 방법 — 검증 데이터에도 fit
wrong = RobustScaler()
wrong.fit_transform(X_train)
X_valid_wrong = wrong.fit_transform(X_valid)   # ⚠️ fit_transform!

print(f"올바름 - 검증 데이터 최댓값: {X_valid_s.max():.4f}")
print(f"잘못됨 - 검증 데이터 최댓값: {X_valid_wrong.max():.4f}")

값이 다릅니다. 잘못된 쪽은 **검증 데이터 자신의 중앙값과 IQR로 변환**되었기 때문입니다.
에러가 나지 않고 조용히 다른 결과를 내기 때문에, 알고 있지 않으면 발견하기 어렵습니다.

같은 규칙이 결측치 대체(`SimpleImputer`)와 인코딩(`OneHotEncoder`)에도 그대로 적용됩니다.
**"학습 데이터에서 배운 것만 검증 데이터에 적용한다"** 가 하나의 원칙입니다.

In [ ]:
# 스케일링 결과는 numpy 배열입니다 (컬럼 이름이 사라짐)
print(type(X_train_s), X_train_s.shape)
print()

# 컬럼 이름을 유지하고 싶다면 DataFrame으로 되돌립니다
X_train_df = pd.DataFrame(X_train_s, columns=X_train.columns, index=X_train.index)
X_train_df.head(3).round(3)

---

## 8. 1부 정리 — `prepare_trips()` 함수

지금까지의 단계를 함수 하나로 묶습니다. **03·04번 노트북에서 이 함수를 그대로 가져다 씁니다.**

In [ ]:
def prepare_trips(raw):
    """[회귀] 택시 데이터 -> (X, y). 예측 대상은 duration(이동 시간, 분)."""
    # ① 이상치 제거
    df = raw[(raw["duration"] > 0) & (raw["speed"] < 60)].copy()

    # ② 불필요한 컬럼 삭제 (결측치 처리보다 먼저!)
    df = df.drop(columns=[
        "pickup", "dropoff",            # datetime — weekday/hour로 대체
        "pickup_zone", "dropoff_zone",  # 고유값 194/203개
        "speed",                        # duration으로 계산 → 데이터 누출
        "total",                        # fare + tip + tolls → 중복
    ])

    # ③ 결측치 제거 (전체의 1.2%)
    df = df.dropna()

    # ④ 범주형 인코딩
    df = pd.get_dummies(df, columns=["color", "payment",
                                     "pickup_borough", "dropoff_borough"],
                        drop_first=True)

    return df.drop(columns="duration"), df["duration"]


X_reg, y_reg = prepare_trips(trips)

print(f"[회귀] X {X_reg.shape}  y {y_reg.shape}")
print("남은 object 컬럼:", X_reg.select_dtypes(include="object").columns.tolist())
print("결측치:", X_reg.isnull().sum().sum())

만든 함수로 분리와 스케일링까지 이어서 해봅니다.
**03·04번 노트북은 여기서 만든 것과 같은 형태의 `X_train`/`X_valid`로 시작합니다.**

In [ ]:
# 분리 + 스케일링까지 한 번에
X_train, X_valid, y_train, y_valid = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=RANDOM_STATE
)

scaler = RobustScaler()
X_train_s = scaler.fit_transform(X_train)
X_valid_s = scaler.transform(X_valid)

print(f"학습 {X_train_s.shape}  검증 {X_valid_s.shape}")
print(f"학습 데이터 범위: {X_train_s.min():.2f} ~ {X_train_s.max():.2f}")
print(f"검증 데이터 범위: {X_valid_s.min():.2f} ~ {X_valid_s.max():.2f}")

**1부 끝.** 택시 데이터는 이제 모델에 바로 넣을 수 있는 상태입니다.

여기까지 판단이 갈리는 대목이 하나도 없었다는 점을 짚어두세요. 시속 4,834마일은 지우는 게
당연했고, 결측 1%도 지우는 게 당연했습니다. **현실의 데이터는 대개 그렇게 친절하지 않습니다.**

이제 타이타닉으로 넘어가, **같은 단계에서 판단이 필요해지는 경우**들을 봅니다.

---

# 2부. 타이타닉 — 판단이 필요한 경우들

**전처리 순서는 1부와 똑같습니다.** 달라지는 것은 각 단계에서 내려야 하는 결정입니다.

| 단계 | 1부 (택시) | 2부 (타이타닉) |
|---|---|---|
| 순서 | 지켜도 티가 안 남 | **틀리면 891행이 182행**이 됨 |
| 이상치 | 도메인 규칙으로 명백 | **IQR로 잡히지만 실제 값** — 지울지 판단 |
| 결측치 | 1%대 → 삭제 | **20~77%** → 채울지 버릴지 |
| 분리 | 회귀라 그냥 나눔 | 분류라 **`stratify`로 클래스 비율 유지** |

## 9. 타이타닉 불러오기

In [ ]:
titanic = sns.load_dataset("titanic")

print("titanic:", titanic.shape)
print("결측치:")
print(titanic.isnull().sum()[lambda s: s > 0])

---

## 10. 순서를 틀리면 어떻게 되는가

1부 1절의 순서표에서 **"② 컬럼 삭제가 ③ 결측치보다 먼저"** 라고 했습니다. 왜 그런지
타이타닉에서 바로 `dropna()`를 호출해 확인해봅시다.

In [ ]:
print("원본                :", titanic.shape[0], "행")
print("바로 dropna()       :", titanic.dropna().shape[0], "행")
print()
print("deck 컬럼 먼저 삭제 후 dropna():",
      titanic.drop(columns="deck").dropna().shape[0], "행")

**891행 → 182행.** 데이터의 80%가 사라졌습니다.

원인은 `deck` 컬럼입니다. 01번에서 봤듯 `deck`은 **77.2%가 비어 있습니다.** `dropna()`는
"한 칸이라도 빈 곳이 있는 행"을 전부 지우므로, 나머지 14개 컬럼이 멀쩡해도 `deck` 하나 때문에
행이 통째로 날아갑니다.

`deck`을 먼저 버리면 712행이 남습니다. **어차피 버릴 컬럼이라면 결측치 처리 전에 버려야 합니다.**

---

## 11. 이상치 ② — IQR 기준과 그 한계

1부에서는 "시속 60마일 이상"처럼 **도메인 규칙**으로 이상치를 걸렀습니다. 그 규칙을 세울 수
없을 때 쓰는 것이 IQR 기준입니다.

도메인 규칙을 세울 수 없을 때(어느 정도가 "너무 비싼" 요금인지 모를 때) 쓰는 방법입니다.
**01번에서 본 박스플롯의 수염이 바로 이 기준입니다.**

```
IQR = Q3 - Q1                    사분위 범위: 가운데 50%가 퍼진 폭
lower_fence = Q1 - 1.5 × IQR     이보다 작으면 이상치
upper_fence = Q3 + 1.5 × IQR     이보다 크면 이상치
```

**왜 하필 1.5인가?** 데이터가 정규분포를 따를 때 이 범위가 전체의 약 99.3%를 담기 때문입니다.
즉 "정상이라면 0.7% 안에서나 나올 값"을 이상치로 봅니다. 더 엄격하게 걸러내려면 1.5 대신
3.0을 쓰기도 합니다.

타이타닉 `fare`에 적용해보겠습니다.

In [ ]:
q1 = titanic["fare"].quantile(0.25)
q3 = titanic["fare"].quantile(0.75)
iqr = q3 - q1

lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

print(f"Q1 (25%)    : {q1:8.4f}")
print(f"Q3 (75%)    : {q3:8.4f}")
print(f"IQR         : {iqr:8.4f}")
print(f"lower_fence : {lower_fence:8.4f}")
print(f"upper_fence : {upper_fence:8.4f}")

is_outlier = (titanic["fare"] < lower_fence) | (titanic["fare"] > upper_fence)
print(f"\n이상치: {is_outlier.sum()}건 ({is_outlier.mean() * 100:.1f}%)")

`lower_fence`가 **-26.7**로 음수입니다. 요금이 음수일 수는 없으니 **아래쪽 이상치는 하나도 없고,
위쪽만 걸러진다**는 뜻입니다. 오른쪽으로 치우친 분포에서 자연스럽게 나타나는 현상입니다.

In [ ]:
titanic_clean = titanic[~is_outlier].copy()   # ~ 는 조건을 뒤집는 연산자(NOT)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=titanic, y="fare", ax=axes[0])
axes[0].set_title(f"제거 전 (n={len(titanic)})")
sns.boxplot(data=titanic_clean, y="fare", ax=axes[1])
axes[1].set_title(f"제거 후 (n={len(titanic_clean)})")
plt.tight_layout()
plt.show()

제거 후에도 **여전히 이상치 점이 보입니다.** 이상하지 않습니다 — 극단값이 빠지면서 Q1, Q3, IQR이
모두 다시 계산되어 fence가 좁아졌기 때문입니다. IQR 기준을 반복 적용하면 데이터가 계속 깎여
나가므로, **보통 한 번만 적용합니다.**

### 지울지 말지 — 판단이 필요한 경우

`titanic`의 `fare` 이상치 116건은 **기록 오류가 아니라 실제 1등급 특실 요금**입니다.
01번 연습문제에서 등급별 요금 박스플롯을 보면, 이상치가 1등급에 몰려 있었습니다.

그렇다면 지우는 게 맞을까요? 지우면 어떤 일이 생기는지 확인해봅시다.

In [ ]:
removed = titanic[is_outlier]

print("제거되는 116건의 구성")
print("  등급별:", removed["pclass"].value_counts().to_dict())
print("  생존율:", f"{removed['survived'].mean() * 100:.1f}%")
print()
print("남는 775건")
print("  등급별:", titanic_clean["pclass"].value_counts().to_dict())
print("  생존율:", f"{titanic_clean['survived'].mean() * 100:.1f}%")
print()
print("전체 생존율:", f"{titanic['survived'].mean() * 100:.1f}%")

제거되는 116건은 **거의 전부 1등급이고 생존율이 훨씬 높습니다.** 이걸 지운다는 것은
**"가장 잘 살아남은 집단"을 통째로 빼고 모델을 만든다**는 뜻입니다.

- **지우는 쪽 논리**: 극단적인 요금이 스케일링과 신경망 학습을 왜곡한다. 남은 775건으로도
  충분히 패턴을 배울 수 있다.
- **남기는 쪽 논리**: 실제 존재한 승객이고, 요금-생존의 관계를 가장 강하게 보여주는 구간이다.
  트리 모델은 이상치에 애초에 둔감하다.

**정답은 없습니다.** 이 시리즈에서는 스케일링과 신경망 실습을 위해 제거하는 쪽을 택하지만,
**"이상치를 제거하면 어떤 집단이 사라지는가"를 확인하고 나서 결정한다**는 절차 자체가 중요합니다.
확인 없이 기계적으로 IQR을 적용하는 것이 가장 위험합니다.

> #### 행을 지우는 세 가지 방법
>
> ```python
> df = df[df["speed"] < 60]                          # ① 남길 조건 (가장 직관적)
> df = df[~is_outlier]                               # ② 뺄 조건을 ~ 로 뒤집기
> df = df.drop(df[is_outlier].index, axis=0)         # ③ 인덱스를 지정해 삭제
> ```
>
> 셋 다 결과는 같습니다. ③은 "이 행들을 지운다"는 의도가 코드에 드러나는 장점이 있습니다.
> 어느 쪽이든 행을 지우면 **인덱스에 구멍이 생깁니다.** 이후 `reset_index(drop=True)`로
> 0부터 다시 매기면 나중에 인덱스로 접근할 때 헷갈리지 않습니다. (`drop=True`를 빼면
> 기존 인덱스가 `index`라는 새 컬럼으로 남아버립니다.)

In [ ]:
titanic_clean = titanic_clean.reset_index(drop=True)
titanic_clean.index[:5]

---

## 12. 결측치 ② — 삭제 대신 대체하기

1부에서는 결측이 1%대라 그냥 지웠습니다. 타이타닉의 `age`는 **19.9%** 입니다. 지우면 승객
177명이 통째로 사라지므로, **채우는 쪽**을 고민하게 됩니다.

먼저 처리하지 않으면 어떻게 되는지부터 봅시다. scikit-learn 모델에 `NaN`을 그대로 넣으면
이런 에러가 납니다.

In [ ]:
from sklearn.linear_model import LinearRegression

X_bad = titanic[["age", "fare"]]     # age에 결측치 177개
y_bad = titanic["survived"]

try:
    LinearRegression().fit(X_bad, y_bad)
except ValueError as e:
    print("ValueError:", str(e)[:150])

### `fillna()` — 무엇으로 채울 것인가

행을 잃고 싶지 않거나 결측치 비율이 클 때 씁니다. **무엇으로 채우느냐**가 핵심입니다.

| 채우는 값 | 언제 | 코드 |
|---|---|---|
| **평균(mean)** | 분포가 좌우대칭일 때 | `df["age"].fillna(df["age"].mean())` |
| **중앙값(median)** | **분포가 치우쳤을 때 (더 안전)** | `df["age"].fillna(df["age"].median())` |
| **최빈값(mode)** | 범주형 | `df["embarked"].fillna(df["embarked"].mode()[0])` |
| 고정값 | 결측이 "없음"을 의미할 때 | `df["deck"].fillna("Unknown")` |
| 앞/뒤 값 | 시계열 | `df["temp"].ffill()` |
| **그룹별 통계** | 집단마다 분포가 다를 때 | 아래 참고 |

01번에서 봤듯 **치우친 분포에서는 평균이 극단값에 끌려갑니다.** 그래서 실무에서는 중앙값을
기본으로 두는 경우가 많습니다.

In [ ]:
age = titanic["age"]

filled_mean = age.fillna(age.mean())
filled_median = age.fillna(age.median())

print(f"원본 (결측 {age.isnull().sum()}개)  평균 {age.mean():.2f}  표준편차 {age.std():.2f}")
print(f"평균으로 대체            평균 {filled_mean.mean():.2f}  표준편차 {filled_mean.std():.2f}")
print(f"중앙값으로 대체          평균 {filled_median.mean():.2f}  표준편차 {filled_median.std():.2f}")

숫자로는 차이가 작아 보이지만, 분포를 그리면 무슨 일이 벌어졌는지 한눈에 보입니다.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.histplot(age.dropna(), bins=30, ax=axes[0])
axes[0].set_title(f"원본 (결측 제외, n={age.notna().sum()})")

sns.histplot(filled_mean, bins=30, ax=axes[1])
axes[1].set_title("평균 29.7로 대체")

sns.histplot(filled_median, bins=30, ax=axes[2])
axes[2].set_title("중앙값 28.0으로 대체")

plt.tight_layout()
plt.show()

가운데와 오른쪽 그래프에서 **한 지점에 막대가 솟아 있습니다.** 177명이 전부 같은 나이가 되었기
때문입니다. 이것이 단순 대체의 부작용입니다.

- 분포의 모양이 인위적으로 바뀝니다
- **표준편차가 줄어듭니다** (14.53 → 평균 대체 13.00, 중앙값 대체 13.02).
  실제보다 "다들 비슷한 나이"인 것처럼 보이게 됩니다

조금 더 나은 방법은 **관련 있는 다른 컬럼으로 그룹을 나눠 채우는 것**입니다.

In [ ]:
# 등급과 성별이 같은 사람들의 중앙값으로 채우기
filled_group = titanic.groupby(["pclass", "sex"])["age"].transform(
    lambda s: s.fillna(s.median())
)

print("등급·성별 그룹의 나이 중앙값")
print(titanic.groupby(["pclass", "sex"])["age"].median().round(1))
print()
print(f"단순 중앙값 대체: 표준편차 {filled_median.std():.2f}")
print(f"그룹별 중앙값 대체: 표준편차 {filled_group.std():.2f}")

1등급 남성의 중앙값은 40세, 3등급 여성은 21.5세로 **그룹마다 크게 다릅니다.** 이 차이를 반영하면
전체를 하나의 값으로 뭉개는 것보다 원래 분포에 가깝게 채울 수 있습니다.

> `transform`과 `mean()`의 차이는 [00번 6절](../00_pandas_for_tabular/00_pandas_for_tabular.ipynb)에서 길이를 직접 찍어보며 확인할 수 있습니다.
>
> `transform`은 그룹별로 계산한 결과를 **원래 행 순서 그대로** 돌려줍니다. `apply`나 `agg`처럼
> 그룹 단위로 축약되지 않아서, 이렇게 컬럼을 통째로 채워 넣을 때 적합합니다.

### 어느 쪽을 택할 것인가

| 상황 | 권장 |
|---|---|
| 결측 비율 < 5% | **삭제** — 간단하고 부작용이 적음 |
| 결측 비율 5~30%, 중요한 변수 | **대체** — 행을 잃기 아까움 |
| 결측 비율 > 50% | **컬럼 삭제** — 채워도 대부분이 추측값 |
| 결측 자체가 정보 | **"결측 여부" 컬럼 추가** 후 대체 |

마지막 항목이 자주 간과됩니다. 예를 들어 소득 설문에서 고소득자가 응답을 회피한다면,
**"비어 있다"는 사실 자체가 예측에 쓸 수 있는 정보**입니다.

```python
df["age_was_missing"] = df["age"].isnull().astype(int)
df["age"] = df["age"].fillna(df["age"].median())
```

이 시리즈에서는 `deck`(77% 결측)은 컬럼 삭제, 나머지는 삭제로 처리합니다.

> #### scikit-learn 방식: `SimpleImputer`
>
> `fillna()`는 pandas 기능이라 학습 데이터의 통계를 검증 데이터에 그대로 적용하기가 번거롭습니다.
> scikit-learn의 `SimpleImputer`는 스케일러와 같은 `fit`/`transform` 구조를 가지고 있어,
> **학습 데이터에서 계산한 중앙값을 검증 데이터에도 동일하게 적용**하기 좋습니다.
>
> ```python
> from sklearn.impute import SimpleImputer
> imputer = SimpleImputer(strategy="median")
> X_train = imputer.fit_transform(X_train)   # 학습 데이터로 중앙값 계산 + 적용
> X_valid = imputer.transform(X_valid)       # 같은 중앙값을 적용
> ```
>
> 왜 이래야 하는지는 7절 "데이터 누출"에서 설명합니다. 결측치 대체도 스케일링과 **똑같은 규칙**을 따릅니다.

---

## 13. 컬럼 정리와 결측치 삭제

타이타닉에서 걷어낼 컬럼은 네 종류입니다. 1부의 택시와 종류는 같고 대상만 다릅니다.

| 종류 | `titanic`에서 | 왜 |
|---|---|---|
| **데이터 누출** | `alive` | `survived`를 yes/no로 바꾼 것뿐 |
| **중복 정보** | `class`(=`pclass`), `embark_town`(=`embarked`), `adult_male`(≈`who`) | 같은 정보 |
| **결측 과다** | `deck` (77.2%) | 채워도 대부분 추측값 |

In [ ]:
titanic_clean = titanic_clean.drop(columns=[
    "alive",                          # survived와 동일 → 데이터 누출
    "class", "embark_town",           # pclass / embarked와 중복
    "deck",                           # 결측치 77%
    "adult_male",                     # who(man/woman/child)와 중복
])

print(titanic_clean.shape)
print(list(titanic_clean.columns))

이제 결측치가 있는 행을 지웁니다. **`deck`을 이미 버렸기 때문에** `age`가 비어 있는 행만 사라집니다.

In [ ]:
print("결측치:")
print(titanic_clean.isnull().sum()[lambda s: s > 0])
print("삭제 전:", titanic_clean.shape)
titanic_clean = titanic_clean.dropna()
print("삭제 후:", titanic_clean.shape)

타이타닉은 **891 → 613행**이 남았습니다. 앞에서 순서를 틀렸을 때의 182행과 비교하면
**3배 이상 차이**입니다. 컬럼 정리를 먼저 한 것만으로 431행을 지킨 셈입니다.

---

## 14. 인코딩과 분리 — 분류에서 달라지는 것

인코딩 방식은 1부와 같습니다(`get_dummies`, `drop_first=True`). **분리 단계에서 하나가
추가되는데, 그것이 `stratify`** 입니다.

In [ ]:
titanic_encoded = pd.get_dummies(titanic_clean, columns=["sex", "embarked", "who"],
                                 drop_first=True)

print("titanic:", titanic_encoded.shape)
print("남은 object 컬럼:", titanic_encoded.select_dtypes(include="object").columns.tolist())
titanic_encoded.head(3)

### `stratify` — 클래스 비율 유지

분류 문제에서 무작위로 나누면 **우연히 한쪽에 특정 클래스가 몰릴 수 있습니다.**
`stratify=y`를 주면 원본의 클래스 비율을 학습·검증 양쪽에 그대로 유지합니다.

타이타닉으로 직접 비교해봅시다.

In [ ]:
Xc = titanic_encoded.drop(columns="survived")
yc = titanic_encoded["survived"]

# stratify 없이
a_tr, a_va, ya_tr, ya_va = train_test_split(Xc, yc, test_size=0.3, random_state=7)

# stratify 적용
b_tr, b_va, yb_tr, yb_va = train_test_split(Xc, yc, test_size=0.3, random_state=7, stratify=yc)

print(f"전체 생존율            : {yc.mean() * 100:.2f}%")
print()
print(f"stratify 없음 - 학습   : {ya_tr.mean() * 100:.2f}%")
print(f"stratify 없음 - 검증   : {ya_va.mean() * 100:.2f}%")
print()
print(f"stratify 적용 - 학습   : {yb_tr.mean() * 100:.2f}%")
print(f"stratify 적용 - 검증   : {yb_va.mean() * 100:.2f}%")

`stratify` 없이 나누면 학습 33.57% / 검증 39.67%로 **6%p나 벌어집니다.**
검증 데이터에 생존자가 실제보다 많이 들어간 셈이라, 성능 평가가 왜곡됩니다.
`stratify=yc`를 주면 35.43% / 35.33%로 전체 비율(35.40%)에 딱 맞습니다.

**클래스가 불균형할수록 이 차이가 커집니다.** 양성이 1%인 사기 탐지 같은 문제라면
`stratify` 없이는 검증 데이터에 양성이 한 건도 안 들어갈 수도 있습니다.

> 회귀 문제에는 `stratify`를 쓰지 않습니다. 연속값에는 "클래스 비율"이라는 개념이 없기 때문입니다.

In [ ]:
X_train_c, X_valid_c = b_tr, b_va
y_train_c, y_valid_c = yb_tr, yb_va

print(f"학습: {X_train_c.shape}  검증: {X_valid_c.shape}")

---

## 15. 2부 정리 — `prepare_titanic()` 함수

1부의 `prepare_trips()`와 **단계 구성이 똑같습니다.** 각 단계에서 내린 판단만 다릅니다.

In [ ]:
def prepare_titanic(raw):
    """[분류] 타이타닉 데이터 -> (X, y). 예측 대상은 survived(생존 여부)."""
    # ① 이상치 제거 (fare, IQR 기준)
    q1, q3 = raw["fare"].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    df = raw[(raw["fare"] >= lower) & (raw["fare"] <= upper)].copy()

    # ② 불필요한 컬럼 삭제
    df = df.drop(columns=[
        "alive",                     # survived와 동일 → 데이터 누출
        "class", "embark_town",      # pclass / embarked와 중복
        "deck",                      # 결측치 77%
        "adult_male",                # who와 중복
    ])

    # ③ 결측치 제거 (age 162건)
    df = df.dropna()

    # ④ 범주형 인코딩
    df = pd.get_dummies(df, columns=["sex", "embarked", "who"], drop_first=True)

    return df.drop(columns="survived"), df["survived"]


X_clf, y_clf = prepare_titanic(titanic)

print(f"[분류] X {X_clf.shape}  y {y_clf.shape}  (생존율 {y_clf.mean() * 100:.1f}%)")
print("남은 object 컬럼:", X_clf.select_dtypes(include="object").columns.tolist())
print("결측치:", X_clf.isnull().sum().sum())

---

## 정리

**1부 — 택시 (판단이 갈리지 않는 경우)**

- **순서**: 이상치 → 컬럼 삭제 → 결측치 → 인코딩 → 분리 → 스케일링
- **이상치**는 도메인 규칙으로 — "9.4마일을 7초에" 같은 물리적으로 불가능한 기록
- **결측치**가 5% 미만이면 삭제로 끝
- **원-핫 인코딩**은 순서 없는 범주에, [레이블 인코딩](../../../glossary.md#label-encoding)은 순서 있는 범주에.
  `drop_first=True`로 더미 변수 함정을 피합니다
- **스케일링은 트리 모델에 불필요, 신경망에 필수.** 이상치가 있으면 `RobustScaler`
- **`fit`은 학습 데이터에만.** 검증 데이터에는 `transform`만 — 스케일러·결측치 대체·인코더 모두 동일

**2부 — 타이타닉 (판단이 필요한 경우)**

- **순서를 틀리면** 891행이 182행이 됩니다. 컬럼 삭제를 먼저 한 것만으로 613행을 지켰습니다
- **IQR 기준**(`Q1-1.5×IQR`, `Q3+1.5×IQR`)은 자동 판별에 쓰지만, 걸린 행이 기록 오류인지
  실제 극단값인지는 **무엇이 사라지는지 확인하고 나서** 결정합니다
- **결측치 대체**는 치우친 분포에 중앙값, 그룹별로 나눠 채우면 원래 분포에 더 가깝습니다.
  50%를 넘으면 컬럼째 버립니다
- **`stratify=y`** 로 클래스 비율을 유지합니다 (분류만)

## 스스로 확인해보기

- [ ] 전처리 순서(이상치 → 컬럼 삭제 → 결측치 → 인코딩 → 분리 → 스케일링)를 말할 수 있다
- [ ] 순서를 바꾸면 남는 행 수가 왜 달라지는지 설명할 수 있다
- [ ] 원-핫 인코딩과 레이블 인코딩을 각각 언제 쓰는지 안다
- [ ] `drop_first=True`가 무엇을 막는지 안다
- [ ] **분리한 뒤에 스케일링**해야 하는 이유를 설명할 수 있다
- [ ] 검증 데이터에 `fit`을 하면 안 되는 이유를 안다 (스케일러·결측치 대체·인코더 모두)
- [ ] 이상치가 있을 때 어떤 스케일러를 쓰는지 안다

## 연습 문제

풀어본 뒤 [02_preprocessing_solutions.ipynb](02_preprocessing_solutions.ipynb)에서 확인하세요.
**문제 1~3은 1부(택시), 문제 4~6은 2부(타이타닉)** 범위입니다.

### 1부 — 택시

**문제 1.** `trips`의 `duration` 컬럼에 IQR 기준을 적용해 `lower_fence`와 `upper_fence`를 구하고,
이상치가 몇 건인지 세세요. 이 이상치들을 제거하는 것이 타당할까요? 제거되는 행들의
`distance` 분포를 확인하고 판단해보세요.

**문제 2.** 아래 코드는 에러 없이 실행되지만 **결과가 잘못되어 있습니다.** 무엇이 문제이고
어떻게 고쳐야 하는지 설명하세요.

```python
scaler = StandardScaler()
X_all = scaler.fit_transform(X_reg)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_all, y_reg, test_size=0.2, random_state=42
)
```

**문제 3.** `trips`에서 `pickup_zone`(고유값 194개)을 버리지 않고 쓰는 방법을 하나 이상
제안해보세요. (힌트: 상위 N개만 남기고 나머지를 `"Other"`로 묶는 방법이 있습니다.
실제로 구현해 컬럼이 몇 개가 되는지 확인해보세요.)

### 2부 — 타이타닉

**문제 4.** 타이타닉 `age`의 결측치를 아래 세 방법으로 채우고, 각각의 평균·표준편차와
히스토그램을 비교하세요.
1. 전체 중앙값
2. `pclass`별 중앙값
3. `pclass` + `sex`별 중앙값

**문제 5.** `titanic`의 `embarked`(결측 2건)를 최빈값으로 채우는 코드를 작성하세요.
`mode()`가 왜 `[0]` 인덱싱을 필요로 하는지도 설명해보세요.

**문제 6.** `prepare_titanic`에서 `drop_first=True`를 `False`로 바꾸면 컬럼이 몇 개 늘어나는지
확인하고, 어떤 컬럼이 추가되는지 나열하세요. 트리 모델을 쓸 계획이라면 어느 쪽이 나을까요?

---

다음 노트북: [03_tree_models.ipynb](../03_tree_models/03_tree_models.ipynb) — 여기서 만든
`X`, `y`로 결정 트리와 랜덤 포레스트를 학습시키고 평가합니다.